In [ ]:
import dataclasses
from functools import partial
from typing import Any

import jax
import jax.numpy as jnp
import numpy as np
from flax.core.frozen_dict import FrozenDict


def get_size(data):
    """Return the size of the dataset."""
    sizes = jax.tree_util.tree_map(lambda arr: len(arr), data)
    return max(jax.tree_util.tree_leaves(sizes))


@partial(jax.jit, static_argnames=('padding',))
def random_crop(img, crop_from, padding):
    """Randomly crop an image.

    Args:
        img: Image to crop.
        crop_from: Coordinates to crop from.
        padding: Padding size.
    """
    padded_img = jnp.pad(img, ((padding, padding), (padding, padding), (0, 0)), mode='edge')
    return jax.lax.dynamic_slice(padded_img, crop_from, img.shape)


@partial(jax.jit, static_argnames=('padding',))
def batched_random_crop(imgs, crop_froms, padding):
    """Batched version of random_crop."""
    return jax.vmap(random_crop, (0, 0, None))(imgs, crop_froms, padding)

In [ ]:
class Dataset(FrozenDict):
    """Dataset class.

    This class supports both regular datasets (i.e., storing both observations and next_observations) and
    compact datasets (i.e., storing only observations). It assumes 'observations' is always present in the keys. If
    'next_observations' is not present, it will be inferred from 'observations' by shifting the indices by 1. In this
    case, set 'valids' appropriately to mask out the last state of each trajectory.
    """

    @classmethod
    def create(cls, freeze=True, **fields):
        """Create a dataset from the fields.

        Args:
            freeze: Whether to freeze the arrays.
            **fields: Keys and values of the dataset.
        """
        data = fields
        assert 'observations' in data
        if freeze:
            jax.tree_util.tree_map(lambda arr: arr.setflags(write=False), data)
        return cls(data)

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.size = get_size(self._dict)
        if 'valids' in self._dict:
            (self.valid_idxs,) = np.nonzero(self['valids'] > 0)

    def get_random_idxs(self, num_idxs):
        """Return `num_idxs` random indices."""
        if 'valids' in self._dict:
            return self.valid_idxs[np.random.randint(len(self.valid_idxs), size=num_idxs)]
        else:
            return np.random.randint(self.size, size=num_idxs)

    def sample(self, batch_size: int, idxs=None):
        """Sample a batch of transitions."""
        if idxs is None:
            idxs = self.get_random_idxs(batch_size)
        return self.get_subset(idxs)

    def get_subset(self, idxs):
        """Return a subset of the dataset given the indices."""
        result = jax.tree_util.tree_map(lambda arr: arr[idxs], self._dict)
        if 'next_observations' not in result:
            result['next_observations'] = self._dict['observations'][np.minimum(idxs + 1, self.size - 1)]
        return result
    

In [ ]:
class ReplayBuffer(Dataset):
    """Replay buffer class.

    This class extends Dataset to support adding transitions.
    """

    @classmethod
    def create(cls, transition, size):
        """Create a replay buffer from the example transition.

        Args:
            transition: Example transition (dict).
            size: Size of the replay buffer.
        """

        def create_buffer(example):
            example = np.array(example)
            return np.zeros((size, *example.shape), dtype=example.dtype)

        buffer_dict = jax.tree_util.tree_map(create_buffer, transition)
        return cls(buffer_dict)

    @classmethod
    def create_from_initial_dataset(cls, init_dataset, size):
        """Create a replay buffer from the initial dataset.

        Args:
            init_dataset: Initial dataset.
            size: Size of the replay buffer.
        """

        def create_buffer(init_buffer):
            buffer = np.zeros((size, *init_buffer.shape[1:]), dtype=init_buffer.dtype)
            buffer[: len(init_buffer)] = init_buffer
            return buffer

        buffer_dict = jax.tree_util.tree_map(create_buffer, init_dataset)
        dataset = cls(buffer_dict)
        dataset.size = dataset.pointer = get_size(init_dataset)
        return dataset

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

        self.max_size = get_size(self._dict)
        self.size = 0
        self.pointer = 0

    def add_transition(self, transition):
        """Add a transition to the replay buffer."""

        def set_idx(buffer, new_element):
            buffer[self.pointer] = new_element

        jax.tree_util.tree_map(set_idx, self._dict, transition)
        self.pointer = (self.pointer + 1) % self.max_size
        self.size = max(self.pointer, self.size)

    def clear(self):
        """Clear the replay buffer."""
        self.size = self.pointer = 0


    def get_episode_boundaries(self):
        """
        Get initial and terminal states for all episodes.
        If athe current online episode has not terminated:
            assign pseudo terminal index to its latest transition for the purpose of sampling from valid trajectories.

        Returns:
            tuple: A tuple containing two arrays:
                - Initial state indices of all episodes.
                - Terminal state indices of all episodes.
        """
        (terminal_indices,) = np.nonzero(self._dict['terminals'])
        previous_pointer = (self.pointer - 1 + self.max_size) % self.max_size

        # if all episodes complete
        if np.isin(previous_pointer, terminal_indices):
            initial_indices = (terminal_indices + 1) % self.max_size
        else:
            # add previous pointer to terminal_indices
            terminal_indices = np.append(terminal_indices, previous_pointer)
            initial_indices = (terminal_indices + 1) % self.max_size

        initial_indices = np.sort(initial_indices)
        terminal_indices = np.sort(terminal_indices)

        return initial_indices, terminal_indices

In [ ]:
import numpy as np
import jax

# Example transition structure
transition_example = {
    "observations": np.array([0.0, 0.1]),
    "actions": np.array([1.0]),
    "rewards": np.array([0.5]),
    "terminals": np.array([0]),
    "next_observations": np.array([0.2, 0.3]),
}

# Replay buffer size
buffer_size = 10

# Create the ReplayBuffer object using the `create` method
replay_buffer = ReplayBuffer.create(transition_example, size=buffer_size)

# Define a function to add transitions and print the buffer state
def add_transition_and_print(replay_buffer, idx):
    transition = {
        "observations": np.array([idx * 1.0, idx * 1.1]),
        "actions": np.array([idx * 2.0]),
        "rewards": np.array([idx * 0.5]),
        "terminals": np.array([1 if idx % 3 == 0 else 0]),  # Mark every 3rd transition as terminal
        "next_observations": np.array([idx * 1.2, idx * 1.3]),
    }
    replay_buffer.add_transition(transition)
    print(f"After adding transition {idx}:")
    print(f"Observations: {replay_buffer._dict['observations'].flatten()} | "
          f"Actions: {replay_buffer._dict['actions'].flatten()} | "
          f"Rewards: {replay_buffer._dict['rewards'].flatten()} | "
          f"Terminals: {replay_buffer._dict['terminals'].flatten()} | "
          f"Next Observations: {replay_buffer._dict['next_observations'].flatten()}")

# Add transitions and print buffer state
for i in range(15):  # Add more transitions than the buffer size
    add_transition_and_print(replay_buffer, i)


In [ ]:
@dataclasses.dataclass
class GCReplayBuffer:
    """Dataset class for goal-conditioned RL.

    This class provides a method to sample a batch of transitions with goals (value_goals and actor_goals) from the
    dataset. The goals are sampled from the current state, future states in the same trajectory, and random states.
    It also supports frame stacking and random-cropping image augmentation.

    It reads the following keys from the config:
    - discount: Discount factor for geometric sampling.
    - value_p_curgoal: Probability of using the current state as the value goal.
    - value_p_trajgoal: Probability of using a future state in the same trajectory as the value goal.
    - value_p_randomgoal: Probability of using a random state as the value goal.
    - value_geom_sample: Whether to use geometric sampling for future value goals.
    - actor_p_curgoal: Probability of using the current state as the actor goal.
    - actor_p_trajgoal: Probability of using a future state in the same trajectory as the actor goal.
    - actor_p_randomgoal: Probability of using a random state as the actor goal.
    - actor_geom_sample: Whether to use geometric sampling for future actor goals.
    - gc_negative: Whether to use '0 if s == g else -1' (True) or '1 if s == g else 0' (False) as the reward.
    - p_aug: Probability of applying image augmentation.

    Attributes:
        dataset: Dataset object.
        config: Configuration dictionary.
    """

    replay_buffer: ReplayBuffer
    config: Any

    def __post_init__(self):
        self.size = self.replay_buffer.size

        # compute trajectory boundaries.
        self.initial_locs, self.terminal_locs = self.replay_buffer.get_episode_boundaries()

        # Assert probabilities sum to 1.
        assert np.isclose(
            self.config['value_p_curgoal'] + self.config['value_p_trajgoal'] + self.config['value_p_randomgoal'], 1.0
        )
        assert np.isclose(
            self.config['actor_p_curgoal'] + self.config['actor_p_trajgoal'] + self.config['actor_p_randomgoal'], 1.0
        )

    def get_final_state_idx(self, idxs):
        """
        Given the initial state idxs, find the idxs of the terminal state
        Indices greater than the highest terminal_idx will wrap around to the lowest terminal_idx because of the circular buffer.
        """
        insertion_points = np.searchsorted(self.terminal_locs, idxs)
        insertion_points[insertion_points == len(self.terminal_locs)] = 0 #since terminal locs is sorted
        final_state_idxs = self.terminal_locs[insertion_points]

        return final_state_idxs


    def sample(self, batch_size: int, idxs=None, evaluation=False):
        """Sample a batch of transitions with goals.

        This method samples a batch of transitions with goals (value_goals and actor_goals) from the dataset. They are
        stored in the keys 'value_goals' and 'actor_goals', respectively. It also computes the 'rewards' and 'masks'
        based on the indices of the goals.

        Args:
            batch_size: Batch size.
            idxs: Indices of the transitions to sample. If None, random indices are sampled.
            evaluafinal_state_idxstion: Whether to sample for evaluation. If True, image augmentation is not applied.
        """

        # trajectory boundaries are dynamic
        self.initial_locs, self.terminal_locs = self.replay_buffer.get_episode_boundaries()

        if idxs is None:
            idxs = self.replay_buffer.get_random_idxs(batch_size)

        batch = self.replay_buffer.sample(batch_size, idxs)

        value_goal_idxs = self.sample_goals(
            idxs,
            self.config['value_p_curgoal'],
            self.config['value_p_trajgoal'],
            self.config['value_p_randomgoal'],
            self.config['value_geom_sample'],
        )
        actor_goal_idxs = self.sample_goals(
            idxs,
            self.config['actor_p_curgoal'],
            self.config['actor_p_trajgoal'],
            self.config['actor_p_randomgoal'],
            self.config['actor_geom_sample'],
        )

        batch['value_goals'] = self.get_observations(value_goal_idxs)
        batch['actor_goals'] = self.get_observations(actor_goal_idxs)
        successes = (idxs == value_goal_idxs).astype(float)
        batch['masks'] = 1.0 - successes
        batch['rewards'] = successes - (1.0 if self.config['gc_negative'] else 0.0)

        if self.config['p_aug'] is not None and not evaluation:
            if np.random.rand() < self.config['p_aug']:
                self.augment(batch, ['observations', 'next_observations', 'value_goals', 'actor_goals'])

        return batch

    def sample_goals(self, idxs, p_curgoal, p_trajgoal, p_randomgoal, geom_sample):
        """Sample goals for the given indices."""
        batch_size = len(idxs)

        # Random goals.
        random_goal_idxs = self.replay_buffer.get_random_idxs(batch_size)

        # Goals from the same trajectory (excluding the current state, unless it is the final state).
        final_state_idxs = self.get_final_state_idx(idxs)

        print("inital idx", idxs)
        print("final idxs", final_state_idxs)
        print("terminal idxs", np.nonzero(self.replay_buffer._dict['terminals'].flatten()))
        # shorten wrapped trajectory to make calculations easier
        final_state_idxs[final_state_idxs < idxs] = self.replay_buffer.max_size - 1

        if geom_sample:
            # Geometric sampling.
            offsets = np.random.geometric(p=1 - self.config['discount'], size=batch_size)  # in [1, inf)
            middle_goal_idxs = np.minimum(idxs + offsets, final_state_idxs)
        else:
            # Uniform sampling.
            distances = np.random.rand(batch_size)  # in [0, 1)
            middle_goal_idxs = np.round(
                (np.minimum(idxs + 1, final_state_idxs) * distances + final_state_idxs * (1 - distances))
            ).astype(int)
        goal_idxs = np.where(
            np.random.rand(batch_size) < p_trajgoal / (1.0 - p_curgoal + 1e-6), middle_goal_idxs, random_goal_idxs
        )

        # Goals at the current state.
        goal_idxs = np.where(np.random.rand(batch_size) < p_curgoal, idxs, goal_idxs)

        return goal_idxs

    def augment(self, batch, keys):
        """Apply image augmentation to the given keys."""
        padding = 3
        batch_size = len(batch[keys[0]])
        crop_froms = np.random.randint(0, 2 * padding + 1, (batch_size, 2))
        crop_froms = np.concatenate([crop_froms, np.zeros((batch_size, 1), dtype=np.int64)], axis=1)
        for key in keys:
            batch[key] = jax.tree_util.tree_map(
                lambda arr: np.array(batched_random_crop(arr, crop_froms, padding)) if len(arr.shape) == 4 else arr,
                batch[key],
            )

    def get_observations(self, idxs):
        """Return the observations for the given indices."""
        return jax.tree_util.tree_map(lambda arr: arr[idxs], self.replay_buffer['observations'])

In [ ]:
import numpy as np
import jax

# Example replay buffer data and configuration for GCReplayBuffer
transition_example = {
    "observations": np.random.rand(15, 3),  # Random observations (e.g., 15 samples with 3 features each)
    "actions": np.random.rand(15, 2),       # Random actions (e.g., 2-dimensional actions)
    "rewards": np.random.rand(15, 1),       # Random rewards
    "terminals": np.zeros(15, dtype=int),   # Terminal flags, all non-terminal initially
    "next_observations": np.random.rand(15, 3),  # Random next observations
}

# Mark some transitions as terminal for testing
transition_example["terminals"][4] = 1
transition_example["terminals"][9] = 1
transition_example["terminals"][14] = 1

# Replay buffer size
buffer_size = 15

# Create the ReplayBuffer object
replay_buffer = ReplayBuffer.create_from_initial_dataset(transition_example, size=buffer_size)

print(replay_buffer)

# Configuration for GCReplayBuffer
config = {
    "discount": 0.99,
    "value_p_curgoal": 0.3,
    "value_p_trajgoal": 0.4,
    "value_p_randomgoal": 0.3,
    "value_geom_sample": True,
    "actor_p_curgoal": 0.4,
    "actor_p_trajgoal": 0.4,
    "actor_p_randomgoal": 0.2,
    "actor_geom_sample": False,
    "gc_negative": True,
    "p_aug": None,  # No augmentation for simplicity
}

# Create GCReplayBuffer object
gc_replay_buffer = GCReplayBuffer(replay_buffer=replay_buffer, config=config)

# Sample goals 10 times
for i in range(10):
    print(f"\nBatch {i + 1}:")
    batch = gc_replay_buffer.sample(batch_size=3)  # Sample a batch of size 3
    for key, value in batch.items():
        print(f"{key}: {value}")


In [18]:
import os
import pickle
import numpy as np

In [19]:
dataset_path = os.path.join("../data_gen_scripts/data", "arm_binpick_easy_randomize_noisy")
with open(f'{dataset_path}.pkl', 'rb') as f:
    offline_train_data = pickle.load(f)

In [16]:
offline_train_data

[{'observation': array([-1.8371022e-01,  5.5920070e-01,  7.0000000e-02,  9.8908716e-01,
          0.0000000e+00,  0.0000000e+00,  1.4733218e-01,  1.5710000e+00,
          7.4199998e-01,  0.0000000e+00, -1.5710000e+00,  0.0000000e+00,
          3.0539999e+00,  1.4490000e+00, -1.3811141e-04,  6.7804295e-01,
          3.7083140e-01,  0.0000000e+00,  0.0000000e+00,  0.0000000e+00,
          7.9999991e-02, -5.8421057e-01,  1.3740630e-01,  5.5244809e-01,
          2.9999999e-02], dtype=float32),
  'next_observation': array([-1.8371022e-01,  5.5920070e-01,  5.7246156e-02,  9.8908716e-01,
          0.0000000e+00,  0.0000000e+00,  1.4733218e-01,  1.5961742e+00,
          6.9770122e-01, -3.2021713e-03, -1.6542430e+00,  2.0709432e-04,
          3.0161474e+00,  1.4104421e+00, -1.4534263e-02,  6.6654694e-01,
          3.6463848e-01, -3.9807844e-01, -2.1636623e-01, -1.6794086e-02,
          8.6975276e-02, -1.5109478e-01,  1.3740630e-01,  5.5244809e-01,
          2.9999999e-02], dtype=float32),
  'ac

In [3]:
len(offline_train_data)

1090958

In [13]:
sum_reward = 0
for i in range(len(offline_train_data)):
    # print(offline_train_data[i]["reward"])
    # print(offline_train_data[i]["terminate"])
    # print(offline_train_data[i]["observation"])
    # print(offline_train_data[i]["traj_id"])
    # print("=======================")
    sum_reward += offline_train_data[i]["reward"]

In [15]:
sum_reward/offline_train_data[-1]["traj_id"]

array([0.7042])

In [24]:
len(offline_train_data)/301

820.0

In [24]:
import os
import pickle
import numpy as np

# Load the original dataset
dataset_path = os.path.join("../data_gen_scripts/data", "arm_binpick_easy_randomize_noisy")
with open(f'{dataset_path}.pkl', 'rb') as f:
    offline_train_data = pickle.load(f)

# Step 1: Find traj_ids to keep
keep_traj_ids = set()
for step in offline_train_data:
    if step['reward'].item() > 0:
        keep_traj_ids.add(step['traj_id'].item())

# Step 2: Filter dataset
filtered_offline_train_data = [
    step for step in offline_train_data
    if step['traj_id'].item() in keep_traj_ids
]

# Step 3: Save the filtered dataset back
filtered_dataset_path = os.path.join("../data_gen_scripts/data", "arm_binpick_easy_randomize_noisy_filtered.pkl")
with open(filtered_dataset_path, 'wb') as f:
    pickle.dump(filtered_offline_train_data, f)

print(f"Filtered dataset saved to {filtered_dataset_path}")


Filtered dataset saved to ../data_gen_scripts/data/arm_binpick_easy_randomize_noisy_filtered.pkl


In [25]:
len(filtered_offline_train_data)

645779